In [31]:
import sys
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import sklearn

In [32]:
# Load the data
Telecom_data = pd.read_csv('TelecomCustomerChurn.csv')

# Telecom Customer Churn Prediction

## Introduction

In this project, we aim to predict customer churn for a telecom company. Customer churn refers to the phenomenon where customers stop using a company's services. Predicting churn is crucial for businesses as it helps them take proactive measures to retain customers and improve their services.

## Dataset

The dataset used in this project is the "Telecom Customer Churn" dataset. It contains information about 7043 customers, including their demographics, account information, and services they have subscribed to. The dataset has 21 columns, each representing a different feature of the customer.

### Features

- `customerID`: Unique identifier for each customer.
- `Gender`: Gender of the customer (Male/Female).
- `SeniorCitizen`: Indicates if the customer is a senior citizen (1) or not (0).
- `Partner`: Indicates if the customer has a partner (Yes/No).
- `Dependents`: Indicates if the customer has dependents (Yes/No).
- `Tenure`: Number of months the customer has stayed with the company.
- `PhoneService`: Indicates if the customer has phone service (Yes/No).
- `MultipleLines`: Indicates if the customer has multiple lines (Yes/No).
- `InternetService`: Type of internet service the customer has (DSL/Fiber optic/No).
- `OnlineSecurity`: Indicates if the customer has online security service (Yes/No).
- `OnlineBackup`: Indicates if the customer has online backup service (Yes/No).
- `DeviceProtection`: Indicates if the customer has device protection service (Yes/No).
- `TechSupport`: Indicates if the customer has tech support service (Yes/No).
- `StreamingTV`: Indicates if the customer has streaming TV service (Yes/No).
- `StreamingMovies`: Indicates if the customer has streaming movies service (Yes/No).
- `Contract`: Type of contract the customer has (Month-to-month/One year/Two year).
- `PaperlessBilling`: Indicates if the customer has paperless billing (Yes/No).
- `PaymentMethod`: Payment method used by the customer (Electronic check/Mailed check/Bank transfer/Credit card).
- `MonthlyCharges`: The amount charged to the customer monthly.
- `TotalCharges`: The total amount charged to the customer.
- `Churn`: Indicates if the customer has churned (Yes/No).

The target variable in this dataset is `Churn`, which we aim to predict using the other features.

## Objective

The objective of this project is to build a machine learning model that can accurately predict whether a customer will churn or not based on their features. This will help the telecom company identify at-risk customers and take necessary actions to retain them.

---

In [33]:
Telecom_data

,customerID,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No,DSL,No,...,No,No,No,No,Monthly,Yes,Manual,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Manual,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Monthly,Yes,Manual,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Monthly,Yes,Manual,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Manual,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No,DSL,Yes,...,No,No,No,No,Monthly,Yes,Manual,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Monthly,Yes,Manual,74.40,306.6,Yes


The dataset imported from kaggle is already clean with no missing values. I will add some noise with the professor noise-functions.

In [35]:
import random

In [36]:
def add_missing(col, amount):
    X = col.copy()
    size = amount if amount >= 1 else int(len(X) * amount)
    indexes = np.random.choice(len(X), size, replace = False )
    X[indexes] = np.nan
    return X

def add_missing_rows(df, amount):
    X = df.copy()
    rows, cols = X.shape
    size = amount if amount >= 1 else int(rows * amount)
    indexes = np.random.choice(rows, size, replace = False ) + 0.5
    for i in indexes:
        X.loc[i] = np.full((cols,),np.nan)
    X = X.sort_index().reset_index(drop=True)
    return X

In [37]:
Telecom_data['TotalCharges'] = add_missing(Telecom_data['TotalCharges'], 0.2)
Telecom_data['MonthlyCharges'] = add_missing(Telecom_data['MonthlyCharges'], 70)
Telecom_data['PhoneService'] = add_missing(Telecom_data['PhoneService'], 100)
Telecom_data['MultipleLines'] = add_missing(Telecom_data['MultipleLines'], 0.1)
Telecom_data['Gender'] = add_missing(Telecom_data['Gender'], 0.7)

In [40]:
Telecom_data.isnull().sum(axis=0)/Telecom_data.shape[0]
# Returns the percentage of missing values in each column

customerID          0.000000
Gender              0.699986
SeniorCitizen       0.000000
Partner             0.000000
Dependents          0.000000
Tenure              0.000000
PhoneService        0.014198
MultipleLines       0.099957
InternetService     0.000000
OnlineSecurity      0.000000
OnlineBackup        0.000000
DeviceProtection    0.000000
TechSupport         0.000000
StreamingTV         0.000000
StreamingMovies     0.000000
Contract            0.000000
PaperlessBilling    0.000000
PaymentMethod       0.000000
MonthlyCharges      0.009939
TotalCharges        0.199915
Churn               0.000000
dtype: float64

Now that the noise has been added, i'll have to drop the <code>Gender</code> column.